# **How Learnable Surrogate Gradients Reshape Trraining Dynamics in Spiking Neural Networks**
### *Mack Werner*
---
Virginia Tech  
*ECE 6524: Final Project*  

---
[LSG (2023)](https://www.ijcai.org/proceedings/2023/0335.pdf)

**Imports & Device**

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import tonic
import tonic.transforms as transforms
import numpy as np
from collections import defaultdict

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

/Users/briccs/speck/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


**LSG SG**

In [2]:
class LSGSpike(torch.autograd.Function):
    @staticmethod
    def forward(ctx, input_tensor, width):
        ctx.save_for_backward(input_tensor, width)
        return (input_tensor > 0).float()

    @staticmethod
    def backward(ctx, grad_output):
        input_tensor, width = ctx.saved_tensors
        mask = (torch.abs(input_tensor) < width / 2).float()
        grad = grad_output * mask / width
        return grad, None

**LIF Layer with LSG Diagnostics**

In [3]:
class LIFLayer(nn.Module):
    def __init__(self, input_size, output_size, beta_init=0.2, threshold=1.0): # beta_init
        super().__init__()
        self.fc = nn.Linear(input_size, output_size)
        self.threshold = threshold

        beta_init = torch.tensor(beta_init)
        self.beta_logit = nn.Parameter(torch.log(beta_init / (1 - beta_init)))

    def forward(self, x, mem):
        beta = torch.sigmoid(self.beta_logit)
        width = 2.0 * self.threshold * torch.sqrt(1.0 + beta ** 2)

        mem = beta * mem + self.fc(x)
        spk = LSGSpike.apply(mem - self.threshold, width)

        near_threshold = (torch.abs(mem - self.threshold) < width / 2).float().mean()
        spike_rate = spk.mean()
        mem_mean = mem.mean()
        mem_std = mem.std()

        stats = {
            "beta": beta.detach(),
            "width": width.detach(),
            "near_threshold": near_threshold.detach(),
            "spike_rate": spike_rate.detach(),
            "mem_mean": mem_mean.detach(),
            "mem_std": mem_std.detach(),
        }

        mem = mem * (1.0 - spk)

        return spk, mem, stats

**SNN Model**

In [4]:
class SNN(nn.Module):
    def __init__(self, input_size=700, hidden_size=256, output_size=20):
        super().__init__()
        self.l1 = LIFLayer(input_size, hidden_size, beta_init=0.2) # **0.2**, 0.5, 0.9
        self.l2 = LIFLayer(hidden_size, hidden_size, beta_init=0.2)
        self.l3 = LIFLayer(hidden_size, hidden_size, beta_init=0.2)
        #self.l4 = LIFLayer(hidden_size, hidden_size, beta_init=0.2)
        self.readout = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        # x shape: [B, T, N]
        B, T, N = x.shape

        mem1 = torch.zeros(B, self.l1.fc.out_features, device=x.device)
        mem2 = torch.zeros(B, self.l2.fc.out_features, device=x.device)
        mem3 = torch.zeros(B, self.l3.fc.out_features, device=x.device)
        #mem4 = torch.zeros(B, self.l4.fc.out_features, device=x.device)

        spk_sum = torch.zeros(B, self.l3.fc.out_features, device=x.device)

        layer_stats = {
            "l1": defaultdict(list),
            "l2": defaultdict(list),
            "l3": defaultdict(list),
       #     "l4": defaultdict(list),
        }

        for t in range(T):
            xt = x[:, t, :]

            spk1, mem1, stats1 = self.l1(xt, mem1)
            spk2, mem2, stats2 = self.l2(spk1, mem2)
            spk3, mem3, stats3 = self.l3(spk2, mem3)
        #    spk4, mem4, stats4 = self.l4(spk3, mem4)

            spk_sum += spk3

            for k, v in stats1.items():
                layer_stats["l1"][k].append(v)
            for k, v in stats2.items():
                layer_stats["l2"][k].append(v)
            for k, v in stats3.items():
                layer_stats["l3"][k].append(v)
            #for k, v in stats4.items():
            #    layer_stats["l4"][k].append(v)

        out = self.readout(spk_sum / T)

        summary_stats = {}
        for layer_name, stats in layer_stats.items():
            summary_stats[layer_name] = {}
            for stat_name, values in stats.items():
                summary_stats[layer_name][stat_name] = torch.stack(values).mean()

        return out, summary_stats

**SHD Dataset**  
[Zenke Lab](https://zenkelab.org/resources/spiking-heidelberg-datasets-shd/)

In [5]:
n_time_bins = 50
batch_size = 32
data_path = "/Users/briccs/Desktop/data"

frame_transform = transforms.ToFrame(
    sensor_size=tonic.datasets.SHD.sensor_size,
    n_time_bins=n_time_bins
)

train_dataset = tonic.datasets.SHD(
    save_to=data_path,
    train=True,
    transform=frame_transform
)

test_dataset = tonic.datasets.SHD(
    save_to=data_path,
    train=False,
    transform=frame_transform
)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

**Model, Loss, Optimizer**

In [6]:
model = SNN(input_size=700, hidden_size=256, output_size=20).to(device) # 64, **256**, 512
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

print(model)

SNN(
  (l1): LIFLayer(
    (fc): Linear(in_features=700, out_features=256, bias=True)
  )
  (l2): LIFLayer(
    (fc): Linear(in_features=256, out_features=256, bias=True)
  )
  (l3): LIFLayer(
    (fc): Linear(in_features=256, out_features=256, bias=True)
  )
  (readout): Linear(in_features=256, out_features=20, bias=True)
)


**Histories**

In [7]:
grad_history = defaultdict(list)
lsg_history = defaultdict(list)

**Training Loop**

In [8]:
num_epochs = 20

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    epoch_grad_sums = defaultdict(float)
    epoch_grad_counts = defaultdict(int)

    epoch_lsg_sums = defaultdict(float)
    epoch_lsg_counts = defaultdict(int)

    for x, y in train_loader:
        x = x.float().to(device)
        y = y.to(device)

        x = x.squeeze(2)  # [B, T, N]

        optimizer.zero_grad()
        out, batch_stats = model(x)
        loss = criterion(out, y)
        loss.backward()

        # Record gradient norm per layer for this batch
        for name, param in model.named_parameters():
            if param.grad is not None and "weight" in name:
                g = param.grad.abs().mean().item()
                epoch_grad_sums[name] += g
                epoch_grad_counts[name] += 1

        # Record LSG diagnostic stats per layer
        for layer_name, stats in batch_stats.items():
            for stat_name, value in stats.items():
                key = f"{layer_name}.{stat_name}"
                epoch_lsg_sums[key] += value.item()
                epoch_lsg_counts[key] += 1

        optimizer.step()

        running_loss += loss.item()
        preds = out.argmax(dim=1)
        correct += (preds == y).sum().item()
        total += y.size(0)

    train_acc = 100.0 * correct / total
    avg_loss = running_loss / len(train_loader)

    for name in epoch_grad_sums:
        avg_grad = epoch_grad_sums[name] / epoch_grad_counts[name]
        grad_history[name].append(avg_grad)

    for key in epoch_lsg_sums:
        avg_value = epoch_lsg_sums[key] / epoch_lsg_counts[key]
        lsg_history[key].append(avg_value)

    
    model.eval()
    test_correct = 0
    test_total = 0

    with torch.no_grad():
        for x, y in test_loader:
            x = x.float().to(device)
            y = y.to(device)

            x = x.squeeze(2)
            out, _ = model(x)

            preds = out.argmax(dim=1)
            test_correct += (preds == y).sum().item()
            test_total += y.size(0)

    test_acc = 100.0 * test_correct / test_total


    print(f"\nEpoch {epoch+1}/{num_epochs} | Loss: {avg_loss:.4f} | Train Acc: {train_acc:.2f}% | Test Acc: {test_acc:.2f}%")

    print("Gradient norms per layer:")
    for name in sorted(grad_history.keys()):
        print(f"  {name}: {grad_history[name][-1]:.6e}")

    print("LSG diagnostic stats:")
    for key in sorted(lsg_history.keys()):
        print(f"  {key}: {lsg_history[key][-1]:.6f}")

KeyboardInterrupt: 